## Sequential Agent

`First_agent.ipynb` and `Multiple_inputs.ipynb` both used a **single node**. Here the graph has **two nodes chained together** with `add_edge`: `first_node` runs, its output becomes part of the state `second_node` sees, and `second_node` builds further on top of it. This is the core building block `SequentialAgent`-style patterns are made of — just wired by hand.

In [ ]:
from typing import TypedDict
from langgraph.graph import StateGraph 

In [ ]:
class AgentState(TypedDict):
    name: str
    age: str
    final: str

def first_node(state: AgentState) -> AgentState:
    """This is the first node of our sequence"""
    state["final"] = f"Hi {state["name"]}"
    return state

def second_node(state: AgentState) -> AgentState:
    """This is the second node of our seuqence"""
    state["final"] = state["final"] + f" You are {state["age"]} years old!"
    return state

**Two nodes sharing one state.** `first_node` writes an initial `final` value from `name`; `second_node` reads that same `final` back out of state and appends to it using `age`. Because both nodes operate on the *same* `AgentState` dict, later nodes can always build on whatever earlier nodes already wrote — that's what makes the chain "sequential" rather than two independent computations.

Same f-string gotcha as before: `f"Hi {state["name"]}"` nests double quotes inside a double-quoted f-string, which only parses on Python 3.12+ (PEP 701).

In [ ]:
graph = StateGraph(AgentState)

graph.add_node("first_node", first_node)
graph.add_node("second_node", second_node)

graph.set_entry_point("first_node")
graph.add_edge("first_node","second_node")
app = graph.compile()

**Wiring the sequence:** `set_entry_point("first_node")` marks where the run starts, and `add_edge("first_node", "second_node")` is what makes execution flow from one node into the next. Note there's no `set_finish_point` / edge to `END` here — LangGraph treats a node with no outgoing edges (`second_node`) as terminal by default, so the graph still finishes correctly. It's usually clearer to be explicit (`graph.add_edge("second_node", END)` or `set_finish_point("second_node")`) once a graph has more than a couple of nodes, so the "exit" is obvious at a glance.

In [ ]:
from IPython.display import Image,display
display(Image(app.get_graph().draw_mermaid_png()))

The rendered diagram should now show two boxes in a line (`first_node → second_node`) instead of one — a quick visual check that the edge actually connects them the way you intended.

In [ ]:
result = app.invoke({"name": "charlie", "age": 20})
result["final"]

The initial `invoke` dict only needs to seed `name` and `age` — `final` isn't provided because no node reads it before writing it. Tracing the run: `first_node` sets `final = "Hi charlie"`, then `second_node` appends to that same key to get `"Hi charlie You are 20 years old!"`. That accumulation-across-nodes is exactly what "sequential" buys you over a single node.